# The warehouse and the model factory: a tour

Run the cells top to bottom (Shift+Enter). Nothing here writes to Postgres or production.

The table map and every column's meaning are in `warehouse/README.md`, and the data-quality IDs (ITM-02, SAL-08, ...)
are in `.ai/extended/data-quality.md`.

**Before you start:** `python -m warehouse.build` has been run since the last production pull.

In [ ]:
# Setup: find the repo root, connect read-only, and make charts readable.
import os, sys, json
from pathlib import Path
root = Path.cwd()
while not (root / 'manage.py').exists():
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
plt.rcParams['figure.figsize'] = (11, 4)

from warehouse.db import connect
con = connect()   # reads the Parquet files, so a rebuild never waits on this notebook
q = lambda sql: con.execute(sql).df()   # q("select ...") returns a table

## 1. The build: when, from which pull, and did the checks pass

A check with `should_be = 0` must be zero or the build fails. The others are for information: how much was filled in,
and what was left out.

In [ ]:
build = json.loads(Path('workspace/warehouse/build.json').read_text())
print('built', build['built_at'], 'from the production pull of', build['source_pull'])
q("select name, count, should_be from checks")

## 2. Every table, with its row count

`warehouse/README.md` has the map of how they connect. The next cell lists any table's columns.

In [ ]:
pd.DataFrame([{'table': t, 'rows': v['rows'], 'built by': v['file']} for t, v in build['tables'].items()])

In [ ]:
table = 'item_outcome'   # change me
q(f"describe {table}")

## 3. One item's whole life

Pick an item that was retagged and then sold. You get its row in `item`, every event in time order, its tag over time,
and its time on the floor.

In [ ]:
item_id = q('''
    select p.item_id from item_price p join floor_interval f using (item_id)
    where p.price_source = 'retag' and f.end_reason = 'sold' limit 1
''').item_id[0]
display(q(f"select item_id, sku, era, category, title, brand, condition, vendor_code, retail, price, cost, sold_at, sold_for from item where item_id = {item_id}"))
display(q(f"select event_at, source, kind, old_value, new_value from item_event where item_id = {item_id} order by event_at"))
display(q(f"select valid_from, valid_to, price, price_source from item_price where item_id = {item_id} order by valid_from nulls first"))
q(f"select start_at, start_source, end_at, end_reason from floor_interval where item_id = {item_id}")

## 4. The floor over time

Items on the floor at each day's end, and their tag value that day (the tag they had then, not today's).
- **Early April:** the V3 retag carry-over.
- **Jumps:** trucks going out.
- **Dips:** sales outrunning processing.

In [ ]:
fd = q("select * from floor_daily order by day")
fig, ax = plt.subplots(1, 2)
ax[0].plot(fd.day, fd['items']); ax[0].plot(fd.day, fd.stale_items); ax[0].set_title('Items on the floor (orange: out over 90 days)')
ax[1].plot(fd.day, fd.tag_value.astype(float)); ax[1].set_title('Tag value on the floor ($)')
fig.autofmt_xdate(); plt.show()

## 5. How fast things sell: sell curves by category

The share of items sold by each day on the floor (Kaplan-Meier: an item still out counts while it's out instead of
being called unsold). Day 43 is Black Friday for stock put out Oct 15, and day 77 is Dec 31.

In [ ]:
cats = ['(all)', 'Toys & games', 'Kitchen & dining', 'Tools & hardware', 'Furniture', 'Outdoor & patio furniture']
sc = q("select * from sell_curve_daily where age <= 120")
for c in cats:
    s = sc[sc.category == c]; plt.step(s.age, s.share_sold, where='post', label=c)
for d in (43, 77): plt.axvline(d, color='grey', lw=0.6, ls='--')
plt.legend(); plt.title('Share sold by days on the floor'); plt.xlabel('days on the floor'); plt.show()
q("pivot (select category, day_mark, share_sold from sell_curve) on day_mark using any_value(share_sold) order by category")

## 6. Supply: weeks of cover by category, last full week

Items on the floor ÷ the last 4 weeks' units sold per week. High cover means the category is stocked deep, and low
cover means it needs buying.

In [ ]:
q('''
    select category, items_on_floor, units_sold, round(revenue) as revenue, weeks_of_cover
    from category_supply
    where week = (select max(week) from category_supply) - 7
    order by revenue desc
''')

## 6b. Trucks: cost, revenue and where the rest went

One row per purchase order (`po_economics`):
- **Revenue:** its items' sales plus the register lines assigned to it.
- **Cost:** spread over the sold items by sale price (`item_cost`).
- **Shrink buckets:** where the data allows.

`still_selling` marks trucks whose revenue is still growing.

In [ ]:
q('''
    select order_number, vendor_code, ordered_date, received_est, round(total_cost) as cost, items, sold, still_out,
           lost_or_scrapped, unknown_fate, never_checked_in, round(revenue) as revenue, revenue_to_cost,
           recovery, sell_through, still_selling
    from po_economics where items > 200 and not cost_unknown
    order by ordered_date desc
''')

## 7. The model factory, step by step

**The question (target `sold_30`):** will an item sell within 30 days of going out? Only items out for at least
30 days are used, so every answer is known.

These cells run the same code `python -m factory.run sold_30` runs (`factory/method.py`), one step at a time.

In [ ]:
from factory import method, run as fr
spec = fr.TARGETS['sold_30']
raw = fr.load('sold_30')
print(len(raw), 'items;', round(raw.y.mean() * 100, 1), '% sold within 30 days')
print('features:', spec['features'])
raw[spec['features'] + ['y']].head()

**Step 1: type each variable.** Categorical, ordinal (condition: salvage < fair < good < very good < like new < new),
continuous, or mostly-null (over 80% missing, so dropped).

In [ ]:
t = method.type_variables(raw, spec['features'], spec['ordinal'])
t.summary()

**Step 2: split the nulls.** A variable with missing values becomes two: `is_<var>_null` (was it missing?) and the
value, filled with the median. Missing then carries its own signal instead of being guessed.

In [ ]:
df = method.split_nulls(raw, t)
print('null flags added:', t.null_flags)

**The final holdout.** Rows are in time order. The last 21 days of items are locked away: no step sees them until
both models are fit. That's the leakage-free check.

In [ ]:
cut = df.start_day.max() - pd.Timedelta(days=fr.HOLDOUT_DAYS - 1)
dev, hold = df[df.start_day < cut], df[df.start_day >= cut]
print(f'development: {len(dev):,} items up to {dev.start_day.max().date()}; holdout: {len(hold):,} items from {cut.date()}')

**Step 3: to uniform through the estimated CDF.** Each continuous or ordinal value becomes its percentile in the
training rows (0 to 1). A $500 item and a $5,000 item then no longer swamp everything else. Here is the tag before and
after:

In [ ]:
from sklearn.preprocessing import QuantileTransformer
qt = QuantileTransformer(n_quantiles=200, output_distribution='uniform').fit(dev[['tag']].fillna(dev.tag.median()))
fig, ax = plt.subplots(1, 2)
ax[0].hist(dev.tag.clip(upper=100), bins=50); ax[0].set_title('tag ($, capped at 100 for the chart)')
ax[1].hist(qt.transform(hold[['tag']].fillna(dev.tag.median())).ravel(), bins=50); ax[1].set_title('tag after the CDF (holdout)')
plt.show()

**Step 4: lasso picks the features, then logistic regression is the baseline.** Lasso (L1) pushes weak features to
exactly zero, and logistic regression is fit on the survivors. The weights are readable: a positive weight means
more likely to sell within 30 days.

In [ ]:
base = method.baseline(t).fit(dev, dev.y)
n_in = len(base.named_steps['prep'].get_feature_names_out()); n_kept = len(base.named_steps['lasso'].keep_)
print(f'lasso kept {n_kept} of {n_in} columns (after one-hot)')
pd.DataFrame(method.explain(base, top=25))

**How good is it, on the holdout it never saw?**
- **AUC:** 0.5 is a coin flip and 1.0 is perfect.
- **Calibration:** when it says 40%, do 40% sell? Each dot is a tenth of the holdout items.

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss
p = base.predict_proba(hold)[:, 1]
print('holdout AUC', round(roc_auc_score(hold.y, p), 3), ' log loss', round(log_loss(hold.y, p), 4),
      ' (always guessing the average:', round(log_loss(hold.y, np.full(len(hold), dev.y.mean())), 4), ')')
cal = pd.DataFrame({'p': p, 'y': hold.y.values}).assign(bin=lambda d: pd.qcut(d.p, 10, duplicates='drop')).groupby('bin', observed=True).mean()
plt.plot([0, 1], [0, 1], color='grey', lw=0.6); plt.scatter(cal.p, cal.y); plt.xlabel('predicted'); plt.ylabel('actually sold in 30 days')
plt.title('Calibration on the holdout'); plt.show()

**Why did it predict this?** The holdout items it was surest would sell, and surest wouldn't:

In [ ]:
h = hold.assign(p=p)[['item_id', 'category', 'brand', 'tag', 'retail', 'same_product_on_floor', 'p', 'y']]
titles = q('select item_id, title from item')
display(h.merge(titles, on='item_id').sort_values('p', ascending=False).head(8))
h.merge(titles, on='item_id').sort_values('p').head(8)

## 8. The full run: baseline against the challenger

Gradient boosting has to beat the baseline by a significant margin on the holdout: the 95% interval of its log-loss
gain must be above zero. If it doesn't, the baseline stays. This takes about a minute.

In [ ]:
board = fr.run('sold_30')
fr._print(board)

## 9. Your turn

Any SQL against the warehouse. A few to start:
- `select category, avg(recovery) from item_outcome where sold group by 1 order by 2 desc`
- `select vendor_code, avg(sold::int) from item_outcome group by 1 having count(*) > 200 order by 2`
- `select * from floor_now where age_band = '91-180' order by price desc limit 20`

In [ ]:
q("select category, round(avg(recovery), 3) as recovery, count(*) as sold from item_outcome where sold group by 1 order by 2 desc")